In [16]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial import distance
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F

In [17]:
n_community = 2
n_members = 3

tokens = []

for ii in range(n_community*n_members+1):
    tokens.append(
        chr(ord('A')+ii)
    )

In [18]:
class brain(nn.Module):
    def __init__(self, input_size, hidden_size, n_centers, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.centroids = nn.Parameter(torch.randn(n_centers, hidden_size) * 0.1)
        # we learn logσ² so σ² = softplus(logvar)
        self.logvar    = nn.Parameter(torch.zeros(n_centers))
        
        self.readout   = nn.Linear(n_centers, output_size)

    def forward(self, x, h0=None):
        # (B, T, D)
        out, h = self.rnn(x, h0)
        
        # compute squared distances (B, T, K)
        diff  = out.unsqueeze(2) - self.centroids.view(1,1,-1,out.size(2))
        dist2 = (diff**2).sum(-1)
        
        # compute φ (B, T, K)
        var   = F.softplus(self.logvar).view(1,1,-1)
        Phi   = torch.exp(-0.5 * dist2 / var)
        
        # predict next token
        B,T,K = Phi.size()
        logits = self.readout(Phi.view(B*T, K)).view(B, T, -1)
        return logits, h

    def repulsion_loss(self, margin=1.0):
        # optional: keep centroids from collapsing on each other
        # pairwise distances between centroids
        C = self.centroids
        D = torch.cdist(C, C, p=2)   # (K, K)
        # we only care i<j
        mask = ~torch.eye(D.size(0), dtype=torch.bool, device=D.device)
        # hinge: repulsion if dist < margin
        repel = F.relu(margin - D[mask]).mean()
        return repel

In [34]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=8):
        
        one_hot_encoded = np.zeros((len(data), len(tokens)), dtype=float)
        for ii, token in enumerate(data):
            one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory, len(tokens)*working_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), len(tokens)))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                for kk in range(working_memory):
                    self.X[ii,jj,kk*len(tokens):(kk+1)*len(tokens)] = \
                    one_hot_encoded[ii+jj+kk,:]
                    
            self.y[ii] = \
                one_hot_encoded[ii+jj+kk+1,:]

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [56]:
### initial training ###
total_samples = 100000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 40
hidden_sleep_size = 10
sleep_output_size = 5
num_layers_wake = 1
num_layers_sleep = 1
output_sleep = len(tokens)
input_size = len(tokens)*working_memory
lr = 4e-4
test_acc = []

data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)#gen_seq(total_samples) #

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

network1 = brain(input_size, hidden_wake_size, n_centers=3, output_size=len(tokens))

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    optimizer.zero_grad()

    if total == 0:
        predicted_y, hidden = network1(X)
    else:
        predicted_y, hidden = network1(X, mem)
    
    # print(predicted_y.shape, y.shape)
    loss = criterion(predicted_y[0], y)
    loss_repel = 1e-1 * network1.repulsion_loss(margin=1.0)
    
    loss = loss + loss_repel
    
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem=hidden.clone()
        true_y = y.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=2)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')


Iter : 1001, loss: 2.1306, accuracy: 0.2070
Iter : 2001, loss: 1.9144, accuracy: 0.2500
Iter : 3001, loss: 1.9968, accuracy: 0.2500
Iter : 4001, loss: 2.2024, accuracy: 0.2640
Iter : 5001, loss: 1.9547, accuracy: 0.2940
Iter : 6001, loss: 1.9337, accuracy: 0.3450
Iter : 7001, loss: 2.2889, accuracy: 0.3600
Iter : 8001, loss: 1.4536, accuracy: 0.3530
Iter : 9001, loss: 2.0704, accuracy: 0.3520
Iter : 10001, loss: 1.9760, accuracy: 0.3670
Iter : 11001, loss: 1.9689, accuracy: 0.3960
Iter : 12001, loss: 1.9906, accuracy: 0.3950
Iter : 13001, loss: 2.3089, accuracy: 0.3900
Iter : 14001, loss: 1.8948, accuracy: 0.4240
Iter : 15001, loss: 1.6176, accuracy: 0.4670
Iter : 16001, loss: 1.6758, accuracy: 0.4360
Iter : 17001, loss: 1.8453, accuracy: 0.4400
Iter : 18001, loss: 2.1721, accuracy: 0.4500
Iter : 19001, loss: 1.7745, accuracy: 0.4050
Iter : 20001, loss: 1.7853, accuracy: 0.4110
Iter : 21001, loss: 1.6655, accuracy: 0.4570
Iter : 22001, loss: 1.3292, accuracy: 0.4400
Iter : 23001, loss:

In [57]:
network1.centroids

Parameter containing:
tensor([[-1.2332e-01,  1.9818e-01, -5.7298e-01,  2.4763e-01, -7.6629e-02,
         -3.0011e-01,  2.6725e-01, -3.8017e-01, -1.2177e-02, -3.1905e-02,
          7.9220e-01,  1.3560e-01,  8.2814e-01,  3.1000e-01, -6.5816e-02,
         -1.2645e-01, -2.0560e-01,  5.8549e-01,  7.0135e-01, -4.7293e-01,
         -3.1502e-01,  8.7847e-01,  9.0271e-02, -9.5254e-01, -7.9129e-01,
         -8.7325e-01,  5.5008e-02,  1.0443e-01, -3.6241e-01,  8.5334e-01,
          2.6311e-01,  4.9865e-02, -2.9178e-01, -1.1683e-02, -2.0703e-01,
         -6.5720e-01,  1.3159e-01,  9.1356e-02,  1.1296e-02,  2.4927e-01],
        [-7.2021e-02, -6.2810e-02,  4.4099e-01,  7.6179e-02,  1.8955e-01,
          3.3370e-01, -5.7593e-02, -1.1355e-01,  2.6308e-02,  1.0593e-02,
         -2.8891e-02, -1.7203e-01,  7.6428e-01,  3.6541e-01, -3.5435e-02,
          3.6405e-01,  1.1290e-02,  8.0867e-01,  4.7976e-01, -7.2824e-01,
         -8.2047e-02,  4.5185e-01,  7.6128e-02, -9.1644e-01, -4.5404e-01,
         -3.323

In [58]:
F.softplus(network1.logvar).view(1,1,-1)

tensor([[[1.4192, 1.0444, 1.3806]]], grad_fn=<ViewBackward0>)